# Mobile Price Classification
## Complete ML Pipeline

This notebook demonstrates a complete machine learning workflow for classifying mobile phone prices into categories based on hardware specifications.

**Dataset**: 2000 mobile phones with 20 features
**Target**: Price range (0=Low, 1=Medium, 2=High, 3=Very High)
**Task**: Multi-class classification

## 1. Setup and Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Add src to path
sys.path.insert(0, '../src')

print("✓ Libraries loaded successfully")

In [ ]:
# Load data
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')
processed_df = pd.read_csv('../data/processed/processed_data.csv')

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nFirst few rows of training data:")
print(train_df.head())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset Info
print("Dataset Information:")
print(processed_df.info())
print(f"\nDataset Shape: {processed_df.shape}")
print(f"\nMemory Usage: {processed_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Statistical Summary
print("Statistical Summary:")
print(processed_df.describe())

In [ ]:
# Check for missing values
missing_values = processed_df.isnull().sum()
if missing_values.sum() == 0:
    print("✓ No missing values found!")
else:
    print("Missing values:")
    print(missing_values[missing_values > 0])

In [ ]:
# Check for duplicates
duplicates = processed_df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")
if duplicates > 0:
    processed_df = processed_df.drop_duplicates()
    print(f"✓ Duplicates removed. New shape: {processed_df.shape}")

In [ ]:
# Target Variable Distribution
print("\nTarget Variable Distribution:")
print(processed_df['price_range'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
processed_df['price_range'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Price Range Distribution')
axes[0].set_xlabel('Price Range')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'], rotation=45)

# Pie chart
processed_df['price_range'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Price Range Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../outputs/figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: class_distribution.png")

## 3. Feature Analysis

In [ ]:
# Correlation Analysis
X = processed_df.drop('price_range', axis=1)
y = processed_df['price_range']

# Calculate correlation with target
correlations = X.corrwith(y).abs().sort_values(ascending=False)

print("\nTop 10 Features by Correlation with Price Range:")
print(correlations.head(10))

# Plot top features
fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Features by Correlation with Price Range')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: feature_correlation.png")

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 12))
correlation_matrix = processed_df.corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: correlation_heatmap.png")

## 4. Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Separate features and target
X = processed_df.drop('price_range', axis=1)
y = processed_df['price_range']

# Split data (80-20 split with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set target distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest set target distribution:")
print(y_test.value_counts().sort_index())

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("✓ Features scaled using StandardScaler")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"\nScaled training data statistics:")
print(X_train_scaled.describe().iloc[:, :5])

## 5. Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print("✓ Models initialized")
print(f"Total models: {len(models)}")
for name in models.keys():
    print(f"  - {name}")

In [ ]:
# Train all models
results = {}

print("\n" + "="*50)
print("Training Models...")
print("="*50)

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict on train and test
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Calculate accuracies
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    
    results[name] = {'train_acc': train_acc, 'test_acc': test_acc}
    
    print(f"{name:20s} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")

print("="*50)

In [ ]:
# Create Results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('test_acc', ascending=False)

print("\nModel Performance Summary (sorted by Test Accuracy):")
print(results_df)

# Save results
os.makedirs('../outputs/results', exist_ok=True)
results_df.to_csv('../outputs/results/model_results.csv')
print("\n✓ Results saved to: outputs/results/model_results.csv")

In [ ]:
# Plot Model Comparison
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind='bar', ax=ax)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.legend(['Train Accuracy', 'Test Accuracy'])
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: model_comparison.png")

## 6. Best Model Evaluation

In [ ]:
# Select Best Model
best_model_name = results_df['test_acc'].idxmax()
best_model = models[best_model_name]
best_accuracy = results_df.loc[best_model_name, 'test_acc']

print(f"\n{'='*50}")
print(f"Best Model: {best_model_name}")
print(f"Test Accuracy: {best_accuracy:.4f}")
print(f"{'='*50}")

# Get predictions from best model
y_pred_best = best_model.predict(X_test_scaled)

In [ ]:
# Classification Report
print(f"\nClassification Report for {best_model_name}:")
print(classification_report(y_test, y_pred_best, 
                          target_names=['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)']))

# Save classification report
report_text = classification_report(y_test, y_pred_best,
                                    target_names=['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'])
with open('../outputs/results/classification_report.txt', 'w') as f:
    f.write(f"Classification Report - {best_model_name}\n")
    f.write(f"{'='*50}\n\n")
    f.write(report_text)

print("✓ Classification report saved")"]}, {
: 
, 
: null, 
: {}, 
: [], 
: [
, 
, 
\nConfusion Matrix for {best_model_name}:")
print(cm)

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax,
            xticklabels=['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'],
            yticklabels=['Low (0)', 'Medium (1)', 'High (2)', 'Very High (3)'])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.savefig('../outputs/figures/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrix.png")

## 7. Feature Importance

In [ ]:
# Check if model has feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f"\nTop 15 Most Important Features ({best_model_name}):")
    print(feature_importance.head(15))
    
    # Plot Feature Importance
    fig, ax = plt.subplots(figsize=(10, 6))
    feature_importance.head(15).plot(x='Feature', y='Importance', kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top 15 Feature Importance - {best_model_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../outputs/figures/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: feature_importance.png")
else:
    print(f"Note: {best_model_name} does not have feature importance attribute")

## 8. Save Best Model

In [ ]:
import pickle

# Save model
model_path = '../models/best_model.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f"✓ Model saved to: {model_path}")
print(f"  Model: {best_model_name}")
print(f"  Test Accuracy: {best_accuracy:.4f}")

# Also save the scaler
scaler_path = '../models/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Scaler saved to: {scaler_path}")

## 9. Summary and Conclusions

In [ ]:
# Create Summary Report
summary = f"""
MOBILE PRICE CLASSIFICATION - PROJECT SUMMARY
{'='*60}

DATASET:
  - Total samples: {len(processed_df)}
  - Training samples: {len(X_train)}
  - Test samples: {len(X_test)}
  - Number of features: {X_train.shape[1]}
  - Number of classes: 4 (Low, Medium, High, Very High)

MODELS EVALUATED:
  - Logistic Regression
  - Decision Tree
  - Random Forest
  - Extra Trees
  - Support Vector Machine (SVM)
  - K-Nearest Neighbors
  - Gradient Boosting

RESULTS:
  - Best Model: {best_model_name}
  - Best Test Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)

TOP 5 MODELS (by Test Accuracy):
{results_df.to_string()}

OUTPUTS GENERATED:
  Figures:
    - class_distribution.png
    - correlation_heatmap.png
    - feature_correlation.png
    - model_comparison.png
    - confusion_matrix.png
    - feature_importance.png

  Results:
    - model_results.csv
    - classification_report.txt

  Models:
    - best_model.pkl
    - scaler.pkl

NEXT STEPS:
  1. Use the saved model for predictions on new data
  2. Fine-tune hyperparameters for better performance
  3. Collect more data to improve model generalization
  4. Deploy the model to production

{'='*60}
"""

print(summary)

# Save summary
with open('../outputs/results/project_summary.txt', 'w') as f:
    f.write(summary)

print("✓ Summary saved to: outputs/results/project_summary.txt")

In [ ]:
# Display all generated outputs
import os

print("\n" + "="*60)
print("PROJECT STRUCTURE - ALL GENERATED FILES")
print("="*60)

for root, dirs, files in os.walk('..'):
    level = root.replace('..', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        if not file.startswith('.'):
            print(f'{subindent}{file}')